# 🔴 Prompt Injection Evaluation - Colab Runner

This notebook runs the **SFT red-team model vs baseline** prompt injection evaluation on Google Colab.

**Prerequisites:**
- Runtime → Change runtime type → **T4 GPU** (for loading the SFT attacker model)
- Upload your project to Google Drive or clone from repo

## Steps:
1. Install dependencies
2. Mount Google Drive & upload project files
3. Set API keys
4. Run evaluation
5. View results

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate peft bitsandbytes
!pip install -q pyyaml requests tqdm sentencepiece tiktoken

## 2. Mount Google Drive & Setup Project

**Option A**: Upload your entire `project/` folder to Google Drive, then mount.

**Option B**: Clone your repo directly (uncomment the git clone cell below).

In [ ]:
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set this to wherever you uploaded your project folder in Google Drive
# For example, if you uploaded to My Drive/11776/project/
PROJECT_DIR = '/content/drive/MyDrive/11776/project'

import os
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# # Option B: Clone from git (uncomment if using)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/project
# PROJECT_DIR = '/content/project'
# import os
# os.chdir(PROJECT_DIR)

## 3. Set API Key

Set your CMU AI Gateway API key. You can also use Colab Secrets (🔑 icon on the left panel).

In [ ]:
import os

# Method 1: Direct (replace with your key)
os.environ['CMU_API_KEY'] = 'sk-KzXTurgBObgHC72_U70XgA'  # <-- Replace with your key

# # Method 2: Use Colab Secrets (recommended, more secure)
# from google.colab import userdata
# os.environ['CMU_API_KEY'] = userdata.get('CMU_API_KEY')

print('CMU_API_KEY set:', 'CMU_API_KEY' in os.environ and len(os.environ['CMU_API_KEY']) > 0)

## 4. Quick API Test

Verify the API is working before running the full experiment.

In [ ]:
import requests, os

API_KEY = os.environ['CMU_API_KEY']
BASE_URL = 'https://ai-gateway.andrew.cmu.edu/v1'

# List available models
r = requests.get(f'{BASE_URL}/models', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=15)
if r.status_code == 200:
    models = [m['id'] for m in r.json().get('data', [])]
    print('Available models:', models)
else:
    print(f'ERROR listing models: {r.status_code}')

# Test chat completion with the defender model
DEFENDER_MODEL = 'llama3-2-90b-instruct'  # Change if needed
resp = requests.post(
    f'{BASE_URL}/chat/completions',
    headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
    json={'model': DEFENDER_MODEL, 'messages': [{'role': 'user', 'content': 'Say hello in one word.'}], 'max_tokens': 10},
    timeout=30
)
if resp.status_code == 200:
    print(f'\n✅ {DEFENDER_MODEL}: {resp.json()["choices"][0]["message"]["content"].strip()}')
else:
    print(f'\n❌ {DEFENDER_MODEL}: HTTP {resp.status_code} - {resp.text[:200]}')

## 5. Check GPU & Adapter

Verify GPU is available and the SFT adapter can be loaded.

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Check adapter files exist
import os
adapter_path = 'final_merged-red_tean/final_adapter'
if os.path.exists(adapter_path):
    files = os.listdir(adapter_path)
    print(f'\nAdapter files: {files}')
    # Check adapter size
    safetensors = os.path.join(adapter_path, 'adapter_model.safetensors')
    if os.path.exists(safetensors):
        size_mb = os.path.getsize(safetensors) / 1e6
        print(f'Adapter size: {size_mb:.1f} MB')
else:
    print(f'\n⚠️ Adapter not found at {adapter_path}')
    print('Make sure your project folder includes final_merged-red_tean/final_adapter/')

## 6. Check Benchmark Data

In [ ]:
import json

test_path = 'data_collection/benchmarks/prompt_injection/test.jsonl'
with open(test_path) as f:
    cases = [json.loads(line) for line in f if line.strip()]

scenarios = {}
diffs = {}
for c in cases:
    scenarios[c['scenario']] = scenarios.get(c['scenario'], 0) + 1
    diffs[c.get('difficulty', 'unknown')] = diffs.get(c.get('difficulty', 'unknown'), 0) + 1

print(f'Total cases: {len(cases)}')
print(f'Scenarios: {scenarios}')
print(f'Difficulties: {diffs}')

## 7. Run Evaluation (SFT Only)

Run the SFT attacker against the defender using `prompt_injection_eval_test.yaml`.

In [ ]:
!python -m redteam_sft.run_prompt_injection_eval \
    --config redteam_sft/prompt_injection_eval_test.yaml

## 8. Run Full Experiment (SFT vs KDA)

Run both attackers (SFT + KDA baseline) against the defender.

In [ ]:
!python -m redteam_sft.run_prompt_injection_eval \
    --config redteam_sft/prompt_injection_eval_kda_vs_sft.yaml

## 9. View Results

In [ ]:
import pandas as pd
import json

# --- Load summary ---
summary_path = 'runs/prompt_injection_eval/summary.csv'
try:
    df_summary = pd.read_csv(summary_path)
    print('=== Summary ===')
    display(df_summary)
except FileNotFoundError:
    # Try the test path
    summary_path = 'runs/prompt_injection_eval_test/summary.csv'
    df_summary = pd.read_csv(summary_path)
    print('=== Summary (test) ===')
    display(df_summary)

In [ ]:
# --- Detailed results analysis ---
import json
import pandas as pd

results_path = 'runs/prompt_injection_eval/results.jsonl'
try:
    with open(results_path) as f:
        results = [json.loads(line) for line in f if line.strip()]
except FileNotFoundError:
    results_path = 'runs/prompt_injection_eval_test/results.jsonl'
    with open(results_path) as f:
        results = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(results)
print(f'Total results: {len(df)}')
print()

# ASR by attacker and scenario
if len(df) > 0:
    print('=== ASR by Attacker × Scenario ===')
    pivot = df.groupby(['attacker_id', 'scenario']).agg(
        cases=('final_label', 'count'),
        successes=('final_label', lambda x: (x == 'success').sum()),
        ASR=('final_label', lambda x: (x == 'success').mean()),
        refusal_rate=('refusal_label', 'mean'),
    ).round(4)
    display(pivot)

    print()
    print('=== ASR by Difficulty ===')
    if 'difficulty' in df.columns:
        diff_pivot = df.groupby(['attacker_id', 'difficulty']).agg(
            ASR=('final_label', lambda x: (x == 'success').mean()),
        ).round(4)
        display(diff_pivot)

## 10. (Optional) Run with --limit for Quick Test

Use `--limit N` to only evaluate N cases for a quick sanity check.

In [ ]:
# Quick test with 5 cases
!python -m redteam_sft.run_prompt_injection_eval \
    --config redteam_sft/prompt_injection_eval_test.yaml \
    --limit 5